# 🖐️ Taller: Gestos con Cámara Web — Control Visual con MediaPipe

**Herramientas:** `mediapipe`, `opencv-python`, `numpy`

---

## 📋 Contenido del taller

1. Instalación de dependencias  
2. Captura de video con OpenCV  
3. Detección de manos con MediaPipe  
4. Contar dedos extendidos  
5. Medir distancia índice-pulgar  
6. Acciones visuales por gesto  
7. **BONUS** — Mini-juego interactivo con la mano  

---
> ⚠️ **Nota:** Este notebook se ejecuta **localmente** (Anaconda / Jupyter). NO funciona en Google Colab porque Colab no tiene acceso a la webcam física del equipo.

## 📦 Celda 1 — Instalación de dependencias

Ejecuta esta celda **una sola vez**. Después puedes saltarla.

In [3]:
# Instala las bibliotecas necesarias en el entorno Conda activo
import sys
!{sys.executable} -m pip install mediapipe==0.10.14 opencv-python numpy --quiet
print("✅ Instalación completada.")

✅ Instalación completada.


In [4]:
import sys
print(sys.version)

3.10.20 | packaged by Anaconda, Inc. | (main, Mar 11 2026, 17:42:35) [MSC v.1942 64 bit (AMD64)]


## 🔍 Celda 2 — Verificar versiones

In [5]:
import cv2
import mediapipe as mp
import numpy as np

print(f"OpenCV  versión : {cv2.__version__}")
print(f"MediaPipe versión: {mp.__version__}")
print(f"NumPy   versión : {np.__version__}")
print("✅ Todo importado correctamente.")

OpenCV  versión : 4.13.0
MediaPipe versión: 0.10.14
NumPy   versión : 2.2.6
✅ Todo importado correctamente.


## 📷 Celda 3 — Prueba básica: capturar video

Abre la webcam y muestra el feed crudo. Presiona **`q`** para salir.

In [6]:
import cv2

cap = cv2.VideoCapture(0)  # 0 = cámara por defecto

if not cap.isOpened():
    print("❌ No se pudo abrir la cámara. Verifica que esté conectada y no en uso.")
else:
    print("✅ Cámara abierta. Presiona 'q' en la ventana para salir.")
    while True:
        ret, frame = cap.read()
        if not ret:
            break
        frame = cv2.flip(frame, 1)  # Espejo horizontal
        cv2.putText(frame, "Presiona 'q' para salir", (10, 30),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 0), 2)
        cv2.imshow("Test Camara", frame)
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

cap.release()
cv2.destroyAllWindows()

✅ Cámara abierta. Presiona 'q' en la ventana para salir.


## 🖐️ Celda 4 — Detección de manos con MediaPipe

Dibuja los 21 puntos de referencia (landmarks) de la mano en tiempo real.

In [8]:
import cv2
import mediapipe as mp

mp_hands = mp.solutions.hands
mp_drawing = mp.solutions.drawing_utils
mp_drawing_styles = mp.solutions.drawing_styles  # Changed from mp_styles to mp_drawing_styles

cap = cv2.VideoCapture(0)

with mp_hands.Hands(
    max_num_hands=2,
    min_detection_confidence=0.7,
    min_tracking_confidence=0.7
) as hands:

    print("✅ Detección activa. Presiona 'q' para salir.")
    while True:
        ret, frame = cap.read()
        if not ret:
            break

        frame = cv2.flip(frame, 1)
        rgb   = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        result = hands.process(rgb)

        if result.multi_hand_landmarks:
            for hand_lms in result.multi_hand_landmarks:
                mp_drawing.draw_landmarks(
                    frame, hand_lms,
                    mp_hands.HAND_CONNECTIONS,
                    mp_drawing_styles.get_default_hand_landmarks_style(),  # Updated variable name
                    mp_drawing_styles.get_default_hand_connections_style()  # Updated variable name
                )

        cv2.imshow("MediaPipe - Landmarks", frame)
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

cap.release()
cv2.destroyAllWindows()

✅ Detección activa. Presiona 'q' para salir.


## ✋ Celda 5 — Contar dedos extendidos

### ¿Cómo funciona?

MediaPipe devuelve **21 landmarks** numerados. Para saber si un dedo está extendido comparamos:

| Dedo    | Punta | Nudillo medio |
|---------|-------|---------------|
| Pulgar  | 4     | 3             |
| Índice  | 8     | 6             |
| Medio   | 12    | 10            |
| Anular  | 16    | 14            |
| Meñique | 20    | 18            |

Para los dedos 2-5: si la **punta** está más arriba que el nudillo → extendido.  
Para el pulgar: comparamos coordenada **x** (dirección lateral).

In [9]:
import cv2
import mediapipe as mp
import numpy as np

mp_hands   = mp.solutions.hands
mp_drawing = mp.solutions.drawing_utils

# ── Índices de puntas y nudillos ────────────────────────────────────────────
TIPS   = [4, 8, 12, 16, 20]   # punta de cada dedo
KNUCKLES = [3, 6, 10, 14, 18]  # nudillo medio de cada dedo

def contar_dedos(lms, handedness="Right"):
    """Devuelve el número de dedos extendidos (0-5)."""
    dedos = []

    # Pulgar: comparación horizontal (x) según mano
    if handedness == "Right":
        dedos.append(1 if lms[4].x < lms[3].x else 0)
    else:
        dedos.append(1 if lms[4].x > lms[3].x else 0)

    # Dedos 2-5: punta más arriba (y menor) que nudillo
    for tip, knk in zip(TIPS[1:], KNUCKLES[1:]):
        dedos.append(1 if lms[tip].y < lms[knk].y else 0)

    return sum(dedos), dedos

# ── Colores según número de dedos ───────────────────────────────────────────
COLORES = {
    0: (50,  50,  50),   # negro
    1: (0,   0,   200),  # rojo
    2: (0,   140, 255),  # naranja
    3: (0,   200, 200),  # amarillo
    4: (0,   200, 0),    # verde
    5: (200, 0,   200),  # magenta
}

cap = cv2.VideoCapture(0)

with mp_hands.Hands(
    max_num_hands=1,
    min_detection_confidence=0.75,
    min_tracking_confidence=0.75
) as hands:

    print("✅ Conteo de dedos activo. Presiona 'q' para salir.")
    while True:
        ret, frame = cap.read()
        if not ret:
            break

        frame  = cv2.flip(frame, 1)
        h, w   = frame.shape[:2]
        rgb    = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        result = hands.process(rgb)

        count = 0
        if result.multi_hand_landmarks:
            hand_lms   = result.multi_hand_landmarks[0]
            handedness = result.multi_handedness[0].classification[0].label
            lms        = hand_lms.landmark

            count, estado = contar_dedos(lms, handedness)
            mp_drawing.draw_landmarks(frame, hand_lms, mp_hands.HAND_CONNECTIONS)

        # Barra lateral con el conteo
        color = COLORES.get(count, (255, 255, 255))
        cv2.rectangle(frame, (w - 120, 0), (w, 80), color, -1)
        cv2.putText(frame, str(count), (w - 90, 65),
                    cv2.FONT_HERSHEY_DUPLEX, 2.5, (255, 255, 255), 3)
        cv2.putText(frame, "dedos", (w - 110, 95),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (200, 200, 200), 1)

        cv2.imshow("Contador de Dedos", frame)
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

cap.release()
cv2.destroyAllWindows()

✅ Conteo de dedos activo. Presiona 'q' para salir.


## 📏 Celda 6 — Distancia índice-pulgar

Mide la distancia euclidiana entre la punta del **pulgar (4)** y la punta del **índice (8)** en píxeles.  
Útil para gestos de "pinch" (pellizco).

In [8]:
import cv2
import mediapipe as mp
import numpy as np
import math

mp_hands   = mp.solutions.hands
mp_drawing = mp.solutions.drawing_utils

def distancia(p1, p2, w, h):
    """Distancia euclidiana entre dos landmarks en píxeles."""
    x1, y1 = int(p1.x * w), int(p1.y * h)
    x2, y2 = int(p2.x * w), int(p2.y * h)
    return math.hypot(x2 - x1, y2 - y1), (x1, y1), (x2, y2)

cap = cv2.VideoCapture(0)

with mp_hands.Hands(max_num_hands=1, min_detection_confidence=0.75) as hands:
    print("✅ Midiendo distancia índice-pulgar. Presiona 'q' para salir.")
    while True:
        ret, frame = cap.read()
        if not ret:
            break

        frame  = cv2.flip(frame, 1)
        h, w   = frame.shape[:2]
        result = hands.process(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))

        if result.multi_hand_landmarks:
            lms = result.multi_hand_landmarks[0].landmark
            mp_drawing.draw_landmarks(frame,
                result.multi_hand_landmarks[0], mp_hands.HAND_CONNECTIONS)

            dist, pt1, pt2 = distancia(lms[4], lms[8], w, h)

            # Dibuja la línea de medición
            cv2.line(frame, pt1, pt2, (255, 165, 0), 3)
            mid = ((pt1[0]+pt2[0])//2, (pt1[1]+pt2[1])//2)
            cv2.circle(frame, mid, 6, (0, 255, 255), -1)

            # Etiqueta con la distancia
            cv2.putText(frame, f"{int(dist)} px", (mid[0]+10, mid[1]-10),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 0), 2)

            # Indicador de pinch
            if dist < 40:
                cv2.putText(frame, "PINCH!", (20, 60),
                            cv2.FONT_HERSHEY_DUPLEX, 1.8, (0, 0, 255), 3)

        cv2.imshow("Distancia Indice-Pulgar", frame)
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

cap.release()
cv2.destroyAllWindows()

✅ Midiendo distancia índice-pulgar. Presiona 'q' para salir.


## 🎨 Celda 7 — Acciones visuales por gesto

| Dedos extendidos | Acción |
|-----------------|--------|
| 0 (puño)        | Fondo negro |
| 1 (índice)      | Fondo azul oscuro |
| 2               | Fondo verde oscuro |
| 3               | Fondo rojo oscuro |
| 4               | Fondo morado |
| 5 (palma)       | fondo blanco |

El objeto (círculo) se mueve siguiendo la punta del índice.

In [11]:
import cv2
import mediapipe as mp
import numpy as np
import math

mp_hands   = mp.solutions.hands
mp_drawing = mp.solutions.drawing_utils

TIPS     = [4, 8, 12, 16, 20]
KNUCKLES = [3, 6, 10, 14, 18]

def contar_dedos(lms, handedness="Right"):
    dedos = []
    if handedness == "Right":
        dedos.append(1 if lms[4].x < lms[3].x else 0)
    else:
        dedos.append(1 if lms[4].x > lms[3].x else 0)
    for tip, knk in zip(TIPS[1:], KNUCKLES[1:]):
        dedos.append(1 if lms[tip].y < lms[knk].y else 0)
    return sum(dedos)

# Escenas: color de fondo BGR, texto de escena
ESCENAS = {
    0: ((30,  30,  30),  "PUNO - Fondo Negro"),
    1: ((100, 20,  20),  "1 DEDO - Fondo Azul"),
    2: ((20,  80,  20),  "2 DEDOS - Fondo Verde"),
    3: ((20,  20,  100), "3 DEDOS - Fondo Rojo"),
    4: ((100, 20,  100), "4 DEDOS - Fondo Morado"),
    5: ((230, 230, 230), "5 DEDOS - Fondo Blanco"),
}

cap    = cv2.VideoCapture(0)
obj_x, obj_y = 320, 240  # posición inicial del objeto

with mp_hands.Hands(max_num_hands=1,
                    min_detection_confidence=0.75,
                    min_tracking_confidence=0.75) as hands:

    print("✅ Control visual activo. Gestos: 0-5 dedos. Presiona 'q' para salir.")
    while True:
        ret, frame = cap.read()
        if not ret:
            break

        frame  = cv2.flip(frame, 1)
        h, w   = frame.shape[:2]
        result = hands.process(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))

        count = -1
        if result.multi_hand_landmarks:
            hand_lms   = result.multi_hand_landmarks[0]
            handedness = result.multi_handedness[0].classification[0].label
            lms        = hand_lms.landmark

            count = contar_dedos(lms, handedness)

            # Mover objeto con la punta del índice
            obj_x = int(lms[8].x * w)
            obj_y = int(lms[8].y * h)

            mp_drawing.draw_landmarks(frame, hand_lms, mp_hands.HAND_CONNECTIONS)

        # Fondo de color
        color_fondo, escena_txt = ESCENAS.get(count, ((50, 50, 50), "Sin mano"))
        overlay = np.full_like(frame, color_fondo, dtype=np.uint8)
        frame   = cv2.addWeighted(frame, 0.55, overlay, 0.45, 0)

        # Objeto que sigue el dedo
        radio = 25
        cv2.circle(frame, (obj_x, obj_y), radio, (0, 220, 255), -1)
        cv2.circle(frame, (obj_x, obj_y), radio, (255, 255, 255), 2)

        # HUD
        cv2.rectangle(frame, (0, 0), (w, 45), (0, 0, 0), -1)
        cv2.putText(frame, escena_txt, (10, 32),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.85, (255, 255, 255), 2)
        if count >= 0:
            cv2.putText(frame, f"Dedos: {count}", (w - 160, 32),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.85, (0, 255, 200), 2)

        cv2.imshow("Control Visual por Gestos", frame)
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

cap.release()
cv2.destroyAllWindows()

✅ Control visual activo. Gestos: 0-5 dedos. Presiona 'q' para salir.


---
## 🎮 Celda 8 — BONUS: Mini-juego "Atrapa las burbujas"

### Reglas
- Aparecen **burbujas** de colores que caen desde arriba.
- Controla una **paleta** con el **dedo índice**.
- **Toca** una burbuja para sumar puntos.
- Usa el gesto de **pinch** (pulgar + índice juntos) para activar **modo turbo** (paleta más grande).
- Las burbujas que se escapan te restan vidas ❤️.
- El juego termina al perder las 3 vidas.

In [10]:
import cv2
import mediapipe as mp
import numpy as np
import math
import random
import time

# ─────────────────────────────────────────────────────────────
#  CONFIGURACIÓN
# ─────────────────────────────────────────────────────────────
ANCHO, ALTO = 800, 600
FPS_TARGET  = 30

mp_hands   = mp.solutions.hands
mp_drawing = mp.solutions.drawing_utils

# ─────────────────────────────────────────────────────────────
#  CLASE BURBUJA
# ─────────────────────────────────────────────────────────────
class Burbuja:
    COLORES = [
        (255, 100, 100), (100, 255, 100), (100, 180, 255),
        (255, 220,  80), (220,  80, 255), (80,  255, 220)
    ]

    def __init__(self, w, nivel=1):
        self.r     = random.randint(18, 38)
        self.x     = random.randint(self.r, w - self.r)
        self.y     = -self.r
        self.vy    = random.uniform(2.5, 4.5) + nivel * 0.4
        self.color = random.choice(self.COLORES)
        self.viva  = True
        self.puntos = max(1, 40 - self.r)   # más pequeña → más puntos

    def update(self):
        self.y += self.vy

    def draw(self, frame):
        cx, cy = int(self.x), int(self.y)
        # Sombra
        cv2.circle(frame, (cx+3, cy+3), self.r, (30, 30, 30), -1)
        # Cuerpo
        cv2.circle(frame, (cx, cy), self.r, self.color, -1)
        # Brillo
        cv2.circle(frame, (cx - self.r//3, cy - self.r//3),
                   self.r//4, (255, 255, 255), -1)
        # Borde
        cv2.circle(frame, (cx, cy), self.r,
                   tuple(min(255, c+60) for c in self.color), 2)

    def colisiona(self, px, py, paleta_r):
        return math.hypot(self.x - px, self.y - py) < self.r + paleta_r

# ─────────────────────────────────────────────────────────────
#  FUNCIONES AUXILIARES
# ─────────────────────────────────────────────────────────────
TIPS     = [4, 8, 12, 16, 20]
KNUCKLES = [3, 6, 10, 14, 18]

def contar_dedos(lms, handedness="Right"):
    dedos = []
    if handedness == "Right":
        dedos.append(1 if lms[4].x < lms[3].x else 0)
    else:
        dedos.append(1 if lms[4].x > lms[3].x else 0)
    for tip, knk in zip(TIPS[1:], KNUCKLES[1:]):
        dedos.append(1 if lms[tip].y < lms[knk].y else 0)
    return sum(dedos)

def pinch_activo(lms, w, h, umbral=45):
    p4 = (lms[4].x * w, lms[4].y * h)
    p8 = (lms[8].x * w, lms[8].y * h)
    return math.hypot(p4[0]-p8[0], p4[1]-p8[1]) < umbral

def dibujar_hud(frame, puntaje, vidas, nivel, turbo, w):
    # Barra superior
    cv2.rectangle(frame, (0, 0), (w, 50), (20, 20, 20), -1)
    cv2.putText(frame, f"Puntos: {puntaje}", (10, 35),
                cv2.FONT_HERSHEY_DUPLEX, 1.0, (0, 255, 200), 2)
    # Vidas
    for i in range(vidas):
        cv2.putText(frame, "O", (w - 40 - i*35, 38),
                    cv2.FONT_HERSHEY_SIMPLEX, 1.0, (0, 80, 255), 2)
    # Nivel
    cv2.putText(frame, f"Nivel {nivel}", (w//2 - 50, 35),
                cv2.FONT_HERSHEY_SIMPLEX, 0.9, (255, 220, 80), 2)
    # Turbo
    if turbo:
        cv2.putText(frame, "TURBO", (10, ALTO - 15),
                    cv2.FONT_HERSHEY_DUPLEX, 0.9, (0, 220, 255), 2)

def pantalla_inicio(frame, w, h):
    overlay = frame.copy()
    cv2.rectangle(overlay, (0, 0), (w, h), (10, 10, 30), -1)
    frame = cv2.addWeighted(frame, 0.3, overlay, 0.7, 0)
    msgs = [
        ("ATRAPA LAS BURBUJAS", 0.9, (0, 255, 200), h//2 - 80),
        ("Mueve el INDICE para controlar la paleta", 0.65, (200,200,200), h//2 - 20),
        ("PINCH = Turbo (paleta grande)", 0.65, (255, 220, 80), h//2 + 30),
        ("Muesta la PALMA para comenzar", 0.75, (0, 200, 255), h//2 + 100),
    ]
    for txt, scale, color, y in msgs:
        tw = cv2.getTextSize(txt, cv2.FONT_HERSHEY_SIMPLEX, scale, 2)[0][0]
        cv2.putText(frame, txt, ((w - tw)//2, y),
                    cv2.FONT_HERSHEY_SIMPLEX, scale, color, 2)
    return frame

def pantalla_game_over(frame, puntaje, w, h):
    overlay = np.zeros_like(frame)
    frame   = cv2.addWeighted(frame, 0.3, overlay, 0.7, 0)
    msgs = [
        ("GAME OVER",             1.5,  (0, 80, 255), h//2 - 60),
        (f"Puntuacion: {puntaje}",  1.0,  (255,255,255), h//2 + 20),
        ("Palma abierta = reiniciar", 0.7, (0, 220, 150), h//2 + 90),
    ]
    for txt, scale, color, y in msgs:
        tw = cv2.getTextSize(txt, cv2.FONT_HERSHEY_SIMPLEX, scale, 2)[0][0]
        cv2.putText(frame, txt, ((w - tw)//2, y),
                    cv2.FONT_HERSHEY_SIMPLEX, scale, color, 2)
    return frame

# ─────────────────────────────────────────────────────────────
#  BUCLE PRINCIPAL DEL JUEGO
# ─────────────────────────────────────────────────────────────
def main():
    cap = cv2.VideoCapture(0)
    cap.set(cv2.CAP_PROP_FRAME_WIDTH,  ANCHO)
    cap.set(cv2.CAP_PROP_FRAME_HEIGHT, ALTO)

    # Estado del juego
    estado      = "inicio"   # inicio | jugando | gameover
    puntaje     = 0
    vidas       = 3
    nivel       = 1
    burbujas    = []
    spawn_timer = 0
    spawn_cada  = 40          # frames entre spawns
    paleta_x, paleta_y = ANCHO//2, ALTO - 60
    paleta_base = 30
    particles   = []          # efectos de explosión

    # Detección de gesto "palma" con cooldown
    palma_cooldown = 0

    with mp_hands.Hands(
        max_num_hands=1,
        min_detection_confidence=0.75,
        min_tracking_confidence=0.75
    ) as hands:

        print("Mini-juego iniciado. Presiona 'q' para salir.")
        frame_count = 0

        while True:
            ret, frame = cap.read()
            if not ret:
                break

            frame = cv2.flip(frame, 1)
            h, w  = frame.shape[:2]
            frame_count += 1

            # Fondo degradado
            bg = np.zeros_like(frame)
            for row in range(h):
                t = row / h
                bg[row] = [
                    int(10  + 20  * t),
                    int(10  + 30  * t),
                    int(40  + 40  * t)
                ]
            frame = cv2.addWeighted(frame, 0.45, bg, 0.55, 0)

            # ── Procesar mano ────────────────────────────────
            result = hands.process(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
            dedos_count = -1
            turbo = False

            if result.multi_hand_landmarks:
                lms_obj    = result.multi_hand_landmarks[0]
                handedness = result.multi_handedness[0].classification[0].label
                lms        = lms_obj.landmark

                dedos_count = contar_dedos(lms, handedness)
                turbo       = pinch_activo(lms, w, h)

                # Posición de la paleta = punta del índice
                paleta_x = int(lms[8].x * w)
                paleta_y = int(lms[8].y * h)

                # Dibujar landmarks sutiles
                mp_drawing.draw_landmarks(
                    frame, lms_obj, mp_hands.HAND_CONNECTIONS,
                    mp_drawing.DrawingSpec(color=(80,200,80), thickness=1, circle_radius=2),
                    mp_drawing.DrawingSpec(color=(40,120,40), thickness=1)
                )

            paleta_r = paleta_base + (20 if turbo else 0)

            # ── Cooldown del gesto palma ─────────────────────
            if palma_cooldown > 0:
                palma_cooldown -= 1

            palma_detectada = (dedos_count == 5 and palma_cooldown == 0)

            # ── MÁQUINA DE ESTADOS ───────────────────────────
            if estado == "inicio":
                frame = pantalla_inicio(frame, w, h)
                if palma_detectada:
                    estado = "jugando"
                    puntaje = 0; vidas = 3; nivel = 1
                    burbujas = []; particles = []
                    palma_cooldown = 60

            elif estado == "gameover":
                frame = pantalla_game_over(frame, puntaje, w, h)
                if palma_detectada:
                    estado = "inicio"
                    palma_cooldown = 60

            elif estado == "jugando":
                # Subir nivel cada 100 puntos
                nivel = 1 + puntaje // 100
                spawn_cada = max(15, 40 - nivel * 3)

                # Spawn de burbujas
                spawn_timer += 1
                if spawn_timer >= spawn_cada:
                    burbujas.append(Burbuja(w, nivel))
                    spawn_timer = 0

                # Actualizar burbujas
                nuevas = []
                for b in burbujas:
                    b.update()

                    # Colisión con paleta
                    if b.colisiona(paleta_x, paleta_y, paleta_r):
                        puntaje += b.puntos
                        # Partículas de explosión
                        for _ in range(8):
                            angle = random.uniform(0, 2*math.pi)
                            speed = random.uniform(3, 8)
                            particles.append({
                                'x': b.x, 'y': b.y,
                                'vx': math.cos(angle)*speed,
                                'vy': math.sin(angle)*speed,
                                'color': b.color,
                                'life': 20
                            })
                    elif b.y - b.r > h:   # salió por abajo
                        vidas -= 1
                        if vidas <= 0:
                            estado = "gameover"
                            palma_cooldown = 90
                    else:
                        nuevas.append(b)
                burbujas = nuevas

                # Dibujar burbujas
                for b in burbujas:
                    b.draw(frame)

                # Partículas
                vivas_p = []
                for p in particles:
                    p['x'] += p['vx']; p['y'] += p['vy']
                    p['life'] -= 1
                    if p['life'] > 0:
                        alpha = p['life'] / 20
                        r = int(4 * alpha)
                        cv2.circle(frame, (int(p['x']), int(p['y'])), max(1, r), p['color'], -1)
                        vivas_p.append(p)
                particles = vivas_p

                # Dibujar paleta
                color_paleta = (0, 220, 255) if turbo else (255, 160, 0)
                cv2.circle(frame, (paleta_x, paleta_y), paleta_r, color_paleta, -1)
                cv2.circle(frame, (paleta_x, paleta_y), paleta_r, (255,255,255), 2)

                dibujar_hud(frame, puntaje, vidas, nivel, turbo, w)

            cv2.imshow("Atrapa las Burbujas", frame)
            if cv2.waitKey(1) & 0xFF == ord('q'):
                break

    cap.release()
    cv2.destroyAllWindows()
    print(f"Juego terminado. Puntaje final: {puntaje}")

main()

Mini-juego iniciado. Presiona 'q' para salir.
Juego terminado. Puntaje final: 78


---
## 📝 Resumen de lo que se exploró

| Celda | Concepto |
|-------|----------|
| 3 | Captura de video con `cv2.VideoCapture` |
| 4 | Detección de landmarks con `mediapipe.solutions.hands` |
| 5 | Lógica de conteo de dedos comparando coordenadas Y |
| 6 | Distancia euclidiana entre puntos clave |
| 7 | Mapeo gesto → acción visual (color de fondo, objeto en pantalla, cambio de escena) |
| 8 | Integración completa en mini-juego con estados, colisiones y efectos |

---
*Taller semana_7_5_gestos_webcam_mediapipe — Python local con Anaconda*